# 03 · Alertas de Global Forest Watch, por celda y mes

**Qué hace.** Le pregunta a la API de Global Forest Watch (GFW) cuántas hectáreas con alerta de disturbio de
la vegetación hubo en cada celda de 5 km y en cada mes, y arma el panel mensual con la tasa respecto del
bosque base.

**Cuándo correrlo.** Después de `02_carga_hansen`. Para actualizar, cada mes: no hay que cambiar nada,
porque por defecto llega hasta el último mes completo.

**Qué lee.** Las tablas `GRILLA` y `BOSQUE_BASE`, y la API de GFW, con la llave de la celda 1.

**Qué escribe**, en `DATASET_DEFORESTACION`:

| Tabla | Qué es |
|---|---|
| `PANEL_DEFORESTACION_COLOMBIA` | Una fila por celda y mes: hectáreas con alerta, bosque remanente, tasa y rezagos |

**Qué tienes que cambiar.** Solo la API key de GFW, en la celda 1; lo demás ya apunta al proyecto del
Observatorio (`prueba-ofr`). Si no tienes API key o se venció (dura un año), el **anexo** al final explica cómo sacarla.

**Cuánto tarda.** Entre 10 y 20 minutos, casi todo esperando a la API.

**Qué mide y qué no.** Una alerta marca un disturbio de la vegetación de alta confianza, detectado por
satélite. No es deforestación oficial: no distingue la causa (tala, incendio, daño natural). Su fuerza es
que es **mensual** y que sigue de cerca la cifra oficial: la correlación anual con el IDEAM fue de +0,971 en
2021-2024.

In [ ]:
# =====================================================================
# 1. CONFIGURACIÓN
#    Ya está llena para el proyecto del Observatorio (prueba-ofr), salvo
#    la API key de GFW: pégala abajo, donde dice <<LLENAR>>.
# =====================================================================
# --- Dónde están las tablas en BigQuery ------------------------------
# Proyecto de Google Cloud del Observatorio (Prueba-OFR). Es el id,
# no el nombre: el que aparece en la consola al elegir el proyecto.
PROYECTO = "prueba-ofr"

# Dataset con las tablas de deforestación que arman los notebooks 01
# a 06 (grilla, paneles y panel municipal).
DATASET_DEFORESTACION = "Cambio_Climatico"

# Ubicación de los datos. Debe ser la misma de las tablas del
# Observatorio (en BigQuery: abre una tabla > Detalles > Ubicación de
# los datos). Las del Observatorio están hoy en US.
UBICACION = "US"

# --- Llave de Global Forest Watch -------------------------------------
# API key de la Data API de GFW. Es gratuita y vence al año; el anexo al
# final de este notebook dice cómo sacarla o renovarla. Este notebook
# queda guardado con la llave adentro: no lo compartas fuera del equipo.
GFW_API_KEY = "<<LLENAR: API key de Global Forest Watch>>"

# --- Para actualizar --------------------------------------------------
# El panel va de FECHA_INICIO al mes anterior a FECHA_FIN. Con FECHA_FIN
# vacía llega hasta el último mes completo. Cada corrida vuelve a bajar
# todo el periodo, porque GFW confirma alertas con semanas de retraso y
# los meses recientes cambian de una descarga a otra.
FECHA_INICIO = "2020-01-01"
FECHA_FIN = ""            # en el trabajo de grado: "2026-08-01"

# --- Qué se consulta --------------------------------------------------
# Son los del trabajo de grado. Cambiarlos cambia el panel.
GFW_DATASET = "gfw_integrated_alerts"   # alertas integradas: GLAD-L, GLAD-S2 y RADD
GFW_VERSION = "latest"
CONFIANZA = ("high", "highest")         # se deja por fuera "nominal", la menos confiable
BOSQUE_MINIMO_HA = 50                   # celdas con menos bosque base quedan fuera
CELDAS_POR_LOTE = 400                   # celdas por consulta (la API acepta hasta 256 KB)
LOTES_EN_PARALELO = 6

# --- Solo si corres el notebook en tu computador ----------------------
# En BigQuery déjalo vacío. En un computador: la ruta al archivo .json
# de la llave de una cuenta de servicio con acceso a estas tablas.
RUTA_LLAVE_BIGQUERY = ""

# Carpeta donde se bajan los archivos mientras corre el notebook.
# Vacío: una carpeta temporal. Si ya los tienes bajados, apúntala ahí
# y no se vuelven a descargar.
CARPETA_DESCARGAS = ""

In [ ]:
# =====================================================================
# 2. INSTALAR PAQUETES
#    En BigQuery instala lo que le falte a la máquina; la primera vez
#    tarda uno o dos minutos. No hay que cambiar nada. En un computador
#    con el entorno del repositorio (requirements_local.txt) se puede
#    saltar.
#    Las versiones fijas son las que se usaron en el trabajo de grado,
#    para que los resultados salgan iguales.
# =====================================================================
%pip install --quiet geopandas==1.1.1 requests folium==0.14.0 branca pandas-gbq

In [ ]:
# =====================================================================
# 3. CONEXIÓN CON BIGQUERY Y FUNCIONES DE APOYO
#    No hay que cambiar nada en esta celda. Se conecta a BigQuery y
#    define las funciones con que el resto del notebook lee y escribe
#    tablas. Es igual en todos los notebooks de esta carpeta.
# =====================================================================
import json
import re
import tempfile
import unicodedata
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from google.cloud import bigquery

NOMBRE_NOTEBOOK = "03_carga_alertas_gfw"

# --- 3.1 ¿Quedó algo sin llenar en la celda 1? --------------------------
# Si queda algún <<LLENAR ...>>, el notebook se detiene aquí y dice cuál.
_pendientes = sorted(n for n, v in list(globals().items())
                     if n.isupper() and isinstance(v, str) and "<<LLENAR" in v)
if _pendientes:
    raise ValueError("Falta llenar en la celda 1: " + ", ".join(_pendientes))

# --- 3.2 Conexión ---------------------------------------------------------
# En BigQuery el notebook entra con la cuenta de quien lo corre y no
# necesita llave. La llave solo se usa si se corre en un computador.
if globals().get("RUTA_LLAVE_BIGQUERY"):
    from google.oauth2 import service_account
    client = bigquery.Client(
        project=PROYECTO,
        credentials=service_account.Credentials.from_service_account_file(RUTA_LLAVE_BIGQUERY))
else:
    client = bigquery.Client(project=PROYECTO)
print(f"Conectado a BigQuery: proyecto {PROYECTO}")

# Carpeta para los archivos que se descargan mientras corre el notebook.
# Es de paso: todo lo que importa termina en tablas de BigQuery.
_carpeta = globals().get("CARPETA_DESCARGAS") or ""
DIR_TRABAJO = Path(_carpeta) if _carpeta else Path(tempfile.gettempdir()) / "ofr_deforestacion"
DIR_TRABAJO.mkdir(parents=True, exist_ok=True)


# --- 3.3 Nombres de tabla -----------------------------------------------
def nombre_completo(tabla, dataset=None):
    """'panel_ideam' -> 'proyecto.dataset.PANEL_IDEAM'.

    Las tablas que crean estos notebooks se nombran en mayúscula
    (GRILLA, PANEL_IDEAM...). Si la tabla ya viene completa
    (proyecto.dataset.tabla, como las del Observatorio), se devuelve igual.
    Si no, se arma con PROYECTO y el dataset indicado; por defecto,
    DATASET_DEFORESTACION.
    """
    if tabla.count(".") == 2:
        return tabla
    return f"{PROYECTO}.{dataset or globals().get('DATASET_DEFORESTACION')}.{tabla.upper()}"


# --- 3.4 Leer -------------------------------------------------------------
_MARCA = re.compile(r"^(\[índice(?: sin nombre)?\] )?(?:Nombre original: «(.*?)»)?")


_LECTURA_RAPIDA = [True]


def leer_sql(sql):
    """Corre una consulta SQL en BigQuery y devuelve un DataFrame de pandas.

    Usa la lectura rápida de BigQuery (Storage Read API). Si no está
    disponible (por ejemplo, porque la cuenta no tiene el rol «BigQuery
    Read Session User»), sigue con la lectura normal, que da el mismo
    resultado pero tarda más en tablas grandes.
    """
    trabajo = client.query(sql)
    trabajo.result()                      # si la consulta tiene un error, sale aquí
    resultado = None
    if _LECTURA_RAPIDA[0]:
        try:
            resultado = trabajo.to_arrow()
        except Exception as e:
            _LECTURA_RAPIDA[0] = False
            print("Aviso: la lectura rápida de BigQuery no está disponible "
                  f"({type(e).__name__}); se usa la lectura normal.")
    if resultado is None:
        resultado = trabajo.to_arrow(create_bqstorage_client=False)
    return resultado.to_pandas(date_as_object=False, coerce_temporal_nanoseconds=True)


def leer_tabla(tabla, columnas=None, dataset=None, filtro=None):
    """Lee una tabla de BigQuery a pandas.

    columnas  lista de columnas a traer; por defecto, todas.
    filtro    condición SQL opcional, p. ej. "anio >= 2021".

    Si la tabla se guardó con guardar_tabla(), devuelve las filas en el
    mismo orden en que se guardaron, las columnas con su nombre original
    (tildes, espacios) y el índice.
    """
    destino = nombre_completo(tabla, dataset)
    esquema = client.get_table(destino).schema
    con_orden = any(c.name == "orden_fila" for c in esquema)
    seleccion = ", ".join(f"`{c}`" for c in columnas) if columnas else "*"
    df = leer_sql(f"SELECT {seleccion} FROM `{destino}`"
                  + (f" WHERE {filtro}" if filtro else "")
                  + (" ORDER BY orden_fila" if con_orden else ""))
    df = df.drop(columns=["orden_fila"], errors="ignore")

    originales, indice, sin_nombre = {}, [], set()
    for campo in esquema:
        marca = _MARCA.match(campo.description or "")
        if marca.group(2):
            originales[campo.name] = marca.group(2)
        if marca.group(1) and campo.name in df.columns:
            indice.append(campo.name)
            if "sin nombre" in marca.group(1):
                sin_nombre.add(campo.name)
    if indice:
        df = df.set_index(indice)
        df.index = df.index.set_names(
            [None if n in sin_nombre else originales.get(n, n) for n in indice])
    return df.rename(columns=originales)


# --- 3.5 Escribir -----------------------------------------------------------
_DATASETS_LISTOS = set()


def _nombre_para_bigquery(nombre, usados):
    """Nombre de columna que BigQuery acepta siempre: letras sin tilde,
    números y guion bajo. Los que ya cumplen se dejan igual."""
    texto = str(nombre)
    if re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]{0,299}", texto) and texto.lower() not in usados:
        usados.add(texto.lower())
        return texto
    limpio = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode()
    limpio = re.sub(r"[^0-9A-Za-z]+", "_", limpio).strip("_").lower() or "columna"
    if limpio[0].isdigit():
        limpio = "c_" + limpio
    limpio, base, k = limpio[:280], limpio[:280], 2
    while limpio.lower() in usados:
        limpio, k = f"{base}_{k}", k + 1
    usados.add(limpio.lower())
    return limpio


def _tipos_para_bigquery(df):
    """Deja cada columna en un tipo que BigQuery reciba sin error:
    categorías a texto, enteros con vacíos a entero, mezclas a texto."""
    for c in df.columns:
        s = df[c]
        if isinstance(s.dtype, pd.CategoricalDtype):
            s = s.astype(object).where(s.notna(), None)
        if s.dtype == object:
            valores = s.dropna()
            tipos = {type(v) for v in valores}
            if not tipos or tipos <= {str}:
                pass
            elif all(isinstance(v, (bool, np.bool_)) for v in valores):
                s = s.astype("boolean")
            elif all(isinstance(v, (int, np.integer)) and not isinstance(v, bool) for v in valores):
                s = s.astype("Int64")
            elif all(isinstance(v, (int, float, np.integer, np.floating)) for v in valores):
                s = s.astype(float)
            else:
                s = s.map(lambda v: None if v is None or v is pd.NA
                          or (isinstance(v, float) and v != v) else str(v))
        df[c] = s
    return df


def guardar_tabla(df, tabla, descripcion, dataset=None, indice=False,
                  descripciones=None, procedencia=None):
    """Escribe `df` en BigQuery. Si la tabla ya existe, la REEMPLAZA entera.

    tabla          nombre corto, p. ej. "panel_ideam".
    descripcion    texto que se ve en BigQuery, en los detalles de la tabla.
    dataset        por defecto, DATASET_DEFORESTACION. Si no existe, se crea.
    indice         True para guardar también el índice de pandas como
                   columna(s); leer_tabla() lo recupera.
    descripciones  {columna: texto}, para documentar columnas en BigQuery.
    procedencia    {"fuente": ..., ...}: si se da, deja una fila en la tabla
                   registro_cargas con qué se cargó, de dónde y cuándo.
    """
    dataset = dataset or globals().get("DATASET_DEFORESTACION")
    destino = nombre_completo(tabla, dataset)
    if dataset not in _DATASETS_LISTOS:
        ds = bigquery.Dataset(f"{PROYECTO}.{dataset}")
        ds.location = globals().get("UBICACION", "US")
        client.create_dataset(ds, exists_ok=True)
        _DATASETS_LISTOS.add(dataset)

    datos = df.copy()
    marcas = {}
    if indice:
        nombres_indice = list(datos.index.names)
        datos = datos.reset_index()
        for k, n in enumerate(nombres_indice):
            columna = datos.columns[k]
            marcas[columna] = "[índice sin nombre] " if n is None else "[índice] "
    else:
        datos = datos.reset_index(drop=True)
    if isinstance(datos.columns, pd.MultiIndex):
        datos.columns = [" | ".join(str(x) for x in c if str(x) != "") for c in datos.columns]

    descripciones = descripciones or {}
    usados, notas, columnas = set(), {}, []
    for c in datos.columns:
        nuevo = _nombre_para_bigquery(c, usados)
        texto = marcas.get(c, "")
        if nuevo != str(c):
            texto += f"Nombre original: «{c}»"
        if descripciones.get(c):
            texto += (". " if texto and not texto.endswith("] ") else "") + descripciones[c]
        notas[nuevo] = texto[:1024]
        columnas.append(nuevo)
    datos.columns = columnas
    datos = _tipos_para_bigquery(datos)
    # BigQuery no guarda el orden de las filas, y algunos cálculos (el
    # muestreo del random forest, por ejemplo) dependen de él. Esta
    # columna lo conserva; leer_tabla() la usa y la quita.
    datos["orden_fila"] = np.arange(len(datos), dtype="int64")
    notas["orden_fila"] = ("Posición de la fila en la tabla original. Sirve para leerla "
                           "en el mismo orden: ORDER BY orden_fila.")

    trabajo = client.load_table_from_dataframe(
        datos, destino,
        job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"))
    trabajo.result()

    t = client.get_table(destino)
    t.description = descripcion
    t.schema = [bigquery.SchemaField(f.name, f.field_type, mode=f.mode,
                                     description=notas.get(f.name) or None,
                                     fields=f.fields) for f in t.schema]
    client.update_table(t, ["description", "schema"])
    print(f"Guardada {destino}: {t.num_rows:,} filas, {len(t.schema)} columnas")

    if procedencia is not None:
        fila = pd.DataFrame([{
            "tabla": tabla.upper(),
            "filas": int(t.num_rows),
            "fuente": str(procedencia.get("fuente", "")),
            "parametros": json.dumps({k: v for k, v in procedencia.items() if k != "fuente"},
                                     ensure_ascii=False, default=str),
            "notebook": NOMBRE_NOTEBOOK,
            "cargado_en": datetime.now(timezone.utc),
        }])
        client.load_table_from_dataframe(
            fila, nombre_completo("registro_cargas", dataset),
            job_config=bigquery.LoadJobConfig(write_disposition="WRITE_APPEND")).result()

## 4. La grilla y el bosque base

Se leen de BigQuery la grilla (notebook 01) y el bosque base de Hansen (notebook 02). A la API se le
pregunta por todas las celdas de la grilla; las que no tienen bosque se descartan después.

In [ ]:
grilla = leer_tabla("grilla")
bosque = leer_tabla("bosque_base")
if not grilla["cell_id"].equals(bosque["cell_id"]):
    raise RuntimeError("bosque_base no corresponde a esta grilla: vuelve a correr el notebook 02.")
grilla["bosque_ha"] = bosque["bosque_base_ha"].values

if FECHA_FIN:
    fecha_fin = FECHA_FIN
else:
    hoy = pd.Timestamp.today()
    fecha_fin = pd.Timestamp(hoy.year, hoy.month, 1).strftime("%Y-%m-%d")
MESES = list(pd.date_range(FECHA_INICIO, fecha_fin, freq="MS")[:-1])   # FECHA_FIN no se incluye
print(f"Grilla: {len(grilla):,} celdas · meses: {len(MESES)} ({MESES[0]:%Y-%m} a {MESES[-1]:%Y-%m})")

## 5. La consulta a GFW

Cada celda se manda a la API como un polígono, y la API devuelve, para esa celda, las hectáreas con alerta
de cada día y nivel de confianza. Las hectáreas las calcula la propia API (`area__ha`), que tiene en cuenta
que el área de un píxel en grados cambia con la latitud.

Dos detalles de la API que explican cómo está escrita la consulta:

- **No acepta el operador `IN`** (responde "Unsupported filter operator: in"); por eso la confianza se filtra
  con `OR`.
- **Los campos llevan el nombre del dataset como prefijo** (`gfw_integrated_alerts__date`); por eso la
  consulta se arma con `GFW_DATASET`, y cambiar de producto es cambiar solo esa variable.

In [ ]:
import geopandas as gpd
import shapely.geometry as sg

LADO_CELDA_M = 5_000
CRS_GRILLA = "EPSG:3116"
URL_GFW = "https://data-api.globalforestwatch.org"

condiciones = " OR ".join(f"{GFW_DATASET}__confidence = '{c}'" for c in CONFIANZA)
SQL_LOTE = f"""
SELECT {GFW_DATASET}__confidence,
       {GFW_DATASET}__date,
       SUM(area__ha) AS ha
FROM data
WHERE {GFW_DATASET}__date >= '{FECHA_INICIO}'
  AND {GFW_DATASET}__date < '{fecha_fin}'
  AND ({condiciones})
GROUP BY {GFW_DATASET}__confidence, {GFW_DATASET}__date
"""
print(SQL_LOTE)

# Cada celda como un rectángulo en grados (la API trabaja en EPSG:4326).
cajas = [sg.box(r.ix * LADO_CELDA_M, r.iy * LADO_CELDA_M, (r.ix + 1) * LADO_CELDA_M, (r.iy + 1) * LADO_CELDA_M)
         for r in grilla.itertuples()]
poligonos = gpd.GeoDataFrame({"cell_id": grilla["cell_id"].values}, geometry=cajas,
                             crs=CRS_GRILLA).to_crs("EPSG:4326")
lotes = [poligonos.iloc[i:i + CELDAS_POR_LOTE] for i in range(0, len(poligonos), CELDAS_POR_LOTE)]
print(f"{len(lotes)} lotes de hasta {CELDAS_POR_LOTE} celdas")

## 6. Bajar las alertas, lote por lote

La API trabaja por encargos: se manda un lote, la API devuelve un número de trabajo, y hay que preguntar
cada pocos segundos hasta que termine (hasta 15 minutos por lote; lo normal es menos de un minuto). Se
procesan seis lotes a la vez.

**Si un lote falla, el notebook se detiene con un error** y dice cuáles fallaron. No sigue llenando con
ceros, porque un panel con ceros falsos parece correcto y no lo es. Cada lote que sí llega se guarda en la
carpeta de trabajo, así que al volver a correr la celda solo se piden los que faltan.

In [ ]:
import hashlib
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests

# El nombre del caché depende de todo lo que cambia la respuesta: si se
# cambia una fecha o la confianza, no se reutilizan lotes viejos.
CLAVE_CACHE = hashlib.sha1(json.dumps([GFW_DATASET, GFW_VERSION, FECHA_INICIO, fecha_fin,
                                       list(CONFIANZA), CELDAS_POR_LOTE]).encode()).hexdigest()[:10]
DIR_LOTES = DIR_TRABAJO / "gfw"
DIR_LOTES.mkdir(parents=True, exist_ok=True)


class LlaveGFWInvalida(Exception):
    pass


def procesar_lote(indice, lote, intentos=3):
    """Manda un lote a la API, espera a que termine y devuelve sus filas."""
    cache = DIR_LOTES / f"lote_{GFW_DATASET}_{CLAVE_CACHE}_{indice:04d}.json"
    if cache.exists():
        return json.loads(cache.read_text())

    fc = json.loads(lote.to_json())
    for f in fc["features"]:
        f.pop("id", None)                  # la API rechaza el campo "id"
    cuerpo = {"feature_collection": fc, "id_field": "cell_id", "sql": SQL_LOTE}
    encabezado = {"x-api-key": GFW_API_KEY}

    ultimo_error = ""
    for intento in range(1, intentos + 1):
        try:
            r = requests.post(f"{URL_GFW}/dataset/{GFW_DATASET}/{GFW_VERSION}/query/batch",
                              json=cuerpo, headers=encabezado, timeout=60)
            if r.status_code in (401, 403):
                raise LlaveGFWInvalida(f"HTTP {r.status_code}")
            r.raise_for_status()
            trabajo = r.json()["data"]["job_id"]
            for _ in range(180):                         # hasta 15 minutos
                time.sleep(5)
                jr = requests.get(f"{URL_GFW}/job/{trabajo}", headers=encabezado, timeout=30)
                jr.raise_for_status()
                datos = jr.json()["data"]
                estado = datos.get("status")
                if estado == "success":
                    resultado = requests.get(datos["download_link"], timeout=120).json()
                    cache.write_text(json.dumps(resultado))
                    return resultado
                if estado in ("failed", "error", "partial_success"):
                    raise RuntimeError(f"la API terminó el trabajo como '{estado}'")
            raise TimeoutError("el trabajo no terminó en 15 minutos")
        except LlaveGFWInvalida:
            raise
        except Exception as e:
            ultimo_error = str(e)[:200]
            print(f"  lote {indice}: intento {intento} de {intentos} falló ({ultimo_error})")
    raise RuntimeError(f"lote {indice}: {ultimo_error}")


resultados, fallidos = [None] * len(lotes), {}
inicio = time.time()
with ThreadPoolExecutor(max_workers=LOTES_EN_PARALELO) as pool:
    futuros = {pool.submit(procesar_lote, i, lote): i for i, lote in enumerate(lotes)}
    for k, futuro in enumerate(as_completed(futuros), 1):
        i = futuros[futuro]
        try:
            resultados[i] = futuro.result()
        except LlaveGFWInvalida as e:
            raise RuntimeError(f"GFW rechazó la API key ({e}). Puede que haya vencido: sigue el anexo "
                               "al final del notebook para sacar una nueva.") from None
        except Exception as e:
            fallidos[i] = str(e)
        if k % 10 == 0 or k == len(lotes):
            print(f"  {k}/{len(lotes)} lotes · {time.time() - inicio:.0f} s")

if fallidos:
    raise RuntimeError(f"Fallaron {len(fallidos)} lotes: {sorted(fallidos)}. Vuelve a correr esta celda: "
                       "los que ya llegaron no se vuelven a pedir.")
print("Todos los lotes llegaron.")

## 7. De alertas diarias a hectáreas por mes

La API devuelve una fila por celda, día y nivel de confianza. Se suman por mes, y queda una tabla ancha:
una fila por celda y una columna por mes (`d_2020_01`, `d_2020_02`, …).

In [ ]:
campo_fecha = f"{GFW_DATASET}__date"
columnas_mes = {f"d_{m.strftime('%Y_%m')}": {} for m in MESES}
celdas_con_alerta = set()
for lote in resultados:
    for item in lote:
        cell_id = item.get("cell_id")
        if cell_id is None:
            continue
        celdas_con_alerta.add(cell_id)
        for fila in item.get("result", []):
            fecha = fila.get(campo_fecha)
            ha = fila.get("ha") or 0.0
            if not fecha:
                continue
            clave = f"d_{fecha[:7].replace('-', '_')}"
            if clave in columnas_mes:
                columnas_mes[clave][cell_id] = columnas_mes[clave].get(cell_id, 0.0) + ha

mensual = pd.DataFrame({"cell_id": sorted(celdas_con_alerta)})
for col, valores in columnas_mes.items():
    mensual[col] = mensual["cell_id"].map(valores).fillna(0.0)

ancho = grilla[["cell_id", "lon", "lat", "departamento", "bosque_ha"]].merge(mensual, on="cell_id", how="left")
cols_mes = list(columnas_mes)
ancho[cols_mes] = ancho[cols_mes].fillna(0.0)
ancho = ancho[ancho["bosque_ha"] > 0].copy()
print(f"Celdas con bosque: {len(ancho):,} · con al menos una alerta: "
      f"{int((ancho[cols_mes].sum(axis=1) > 0).sum()):,} · hectáreas con alerta: {ancho[cols_mes].sum().sum():,.0f}")

## 8. El panel: una fila por celda y mes

Cuatro pasos, en este orden:

1. **De ancho a largo:** cada columna de mes pasa a ser una fila (`periodo`, `area_def_ha`).
2. **Filtro de bosque:** se quedan las celdas con al menos 50 ha de bosque base. Una celda de páramo o de
   sabana no se puede deforestar, y solo agregaría ceros.
3. **Panel balanceado:** cada celda tiene una fila en **todos** los meses, también los que no tuvieron alerta
   (con 0). Si faltaran esos meses, un modelo no aprendería qué tan común es no tener alertas.
4. **Variables derivadas**, celda por celda y en orden de tiempo:
   - `def_acum_ha`: hectáreas con alerta acumuladas hasta ese mes.
   - `bosque_remanente_ha`: bosque base menos lo perdido **antes** de ese mes. Es el denominador correcto de
     la tasa: una celda ya agotada no puede perder lo que ya perdió.
   - `tasa_def`: hectáreas del mes sobre el bosque remanente (máximo 1).
   - `evento`: 1 si hubo alguna alerta ese mes.
   - `lag_1_ha` a `lag_3_ha`: las hectáreas de 1, 2 y 3 meses antes, y `media_movil_3`, su promedio.

Al final se agrega el municipio de cada celda, de la grilla.

In [ ]:
largo = ancho.melt(id_vars=["cell_id", "lon", "lat", "bosque_ha", "departamento"],
                   value_vars=cols_mes, var_name="periodo", value_name="area_def_ha")
largo["periodo"] = pd.to_datetime(largo["periodo"].str.replace("d_", "", regex=False), format="%Y_%m")
largo["area_def_ha"] = pd.to_numeric(largo["area_def_ha"], errors="coerce").fillna(0.0)
largo = largo.rename(columns={"bosque_ha": "bosque_base_ha"})

antes = largo["cell_id"].nunique()
largo = largo[largo["bosque_base_ha"] >= BOSQUE_MINIMO_HA].copy()
print(f"Celdas con al menos {BOSQUE_MINIMO_HA} ha de bosque: {largo['cell_id'].nunique():,} de {antes:,}")

fijas = largo.groupby("cell_id")[["lon", "lat", "bosque_base_ha", "departamento"]].first()
todas = pd.MultiIndex.from_product([sorted(largo["cell_id"].unique()), sorted(largo["periodo"].unique())],
                                   names=["cell_id", "periodo"])
panel = (largo.set_index(["cell_id", "periodo"])[["area_def_ha"]]
              .reindex(todas).fillna({"area_def_ha": 0.0})
              .join(fijas, on="cell_id").reset_index()
              .sort_values(["cell_id", "periodo"]))

panel = panel.sort_values(["cell_id", "periodo"]).copy()
panel["def_acum_ha"] = panel.groupby("cell_id", sort=False)["area_def_ha"].cumsum()
previo = panel.groupby("cell_id", sort=False)["def_acum_ha"].shift(1).fillna(0)
panel["bosque_remanente_ha"] = (panel["bosque_base_ha"] - previo).clip(lower=0)
panel["tasa_def"] = panel["area_def_ha"] / panel["bosque_remanente_ha"].replace(0, pd.NA)
panel["tasa_def"] = panel["tasa_def"].fillna(0).clip(upper=1)
panel["evento"] = (panel["area_def_ha"] > 0).astype("int8")
por_celda = panel.groupby("cell_id", sort=False)["area_def_ha"]
for k in (1, 2, 3):
    panel[f"lag_{k}_ha"] = por_celda.shift(k)
panel["media_movil_3"] = por_celda.transform(lambda s: s.shift(1).rolling(3).mean())
panel["anio"] = panel["periodo"].dt.year
panel["mes"] = panel["periodo"].dt.month
panel = panel.merge(grilla[["cell_id", "cod_dane", "municipio"]], on="cell_id", how="left")
panel = panel.reset_index(drop=True)

print(f"Panel: {len(panel):,} filas = {panel.cell_id.nunique():,} celdas x {panel.periodo.nunique()} meses "
      f"({panel.periodo.min():%Y-%m} a {panel.periodo.max():%Y-%m})")
print(f"Meses con alerta: {100 * panel.evento.mean():.2f} % · hectáreas con alerta: {panel.area_def_ha.sum():,.0f}")
print("Hectáreas con alerta por año:")
print(panel.groupby("anio")["area_def_ha"].sum().round(0).to_string())

## 9. Guardar en BigQuery

In [ ]:
guardar_tabla(
    panel, "panel_deforestacion_colombia",
    f"Alertas de disturbio de la vegetación de Global Forest Watch ({GFW_DATASET}, confianza "
    f"{' y '.join(CONFIANZA)}) por celda de 5 km y mes, {MESES[0]:%Y-%m} a {MESES[-1]:%Y-%m}. No es "
    f"deforestación oficial: no distingue la causa. Solo celdas con al menos {BOSQUE_MINIMO_HA} ha de bosque base.",
    descripciones={
        "cell_id": "Identificador de la celda (tabla GRILLA).",
        "periodo": "Primer día del mes.",
        "area_def_ha": "Hectáreas con alerta de alta confianza en el mes.",
        "lon": "Longitud del centro de la celda.",
        "lat": "Latitud del centro de la celda.",
        "bosque_base_ha": "Bosque al empezar el análisis, según Hansen (tabla BOSQUE_BASE).",
        "departamento": "Departamento del centro de la celda.",
        "def_acum_ha": "Hectáreas con alerta acumuladas desde el primer mes hasta este, incluido.",
        "bosque_remanente_ha": "Bosque base menos lo perdido antes de este mes. Denominador de tasa_def.",
        "tasa_def": "area_def_ha / bosque_remanente_ha, con máximo 1.",
        "evento": "1 si el mes tuvo alguna alerta, 0 si no.",
        "lag_1_ha": "Hectáreas con alerta del mes anterior en la misma celda.",
        "lag_2_ha": "Hectáreas con alerta de dos meses antes.",
        "lag_3_ha": "Hectáreas con alerta de tres meses antes.",
        "media_movil_3": "Promedio de las hectáreas con alerta de los tres meses anteriores (sin el actual).",
        "anio": "Año del periodo.",
        "mes": "Mes del periodo (1 a 12).",
        "cod_dane": "Código DANE del municipio del centro de la celda.",
        "municipio": "Nombre del municipio, solo para leer.",
    },
    procedencia={"fuente": f"Global Forest Watch Data API, {GFW_DATASET} ({GFW_VERSION})",
                 "desde": FECHA_INICIO, "hasta_sin_incluir": fecha_fin, "confianza": list(CONFIANZA),
                 "bosque_minimo_ha": BOSQUE_MINIMO_HA})

## 10. Mapa rápido (opcional)

Un vistazo al panel: un mapa de calor de las hectáreas con alerta acumuladas en todo el periodo y, para
hacer clic, los 500 focos más grandes. El mapa base es el gris claro de Esri, que no pide llave, atenuado
para que resalten los datos. Esta celda no escribe nada; si no te interesa el mapa, sáltala.

In [ ]:
import branca.colormap as cm
import folium
from folium.plugins import Fullscreen, HeatMap, MarkerCluster

FOCOS_EN_EL_MAPA = 500

celdas = panel.groupby("cell_id", sort=False)[["lon", "lat", "departamento", "bosque_base_ha"]].first()
celdas = celdas.join(panel.groupby("cell_id", sort=False).agg(
    area_def_total_ha=("area_def_ha", "sum"), meses_con_evento=("evento", "sum")))
celdas = celdas.join(panel.loc[panel.evento == 1].groupby("cell_id")["periodo"].max().rename("ultimo_evento"))
celdas = celdas.reset_index()
celdas["tasa_total"] = celdas["area_def_total_ha"] / celdas["bosque_base_ha"].replace(0, np.nan)
con_def = celdas[celdas["area_def_total_ha"] > 0]

# Mapa base: el gris claro de Esri, que no pide llave. OpenStreetMap y
# CARTO no sirven en BigQuery: el primero bloquea los mapas que no dicen
# desde qué página se piden y el segundo ya exige API key.
mapa = folium.Map(location=(4.2, -73.5), zoom_start=6, control_scale=True, prefer_canvas=True,
                  tiles="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/"
                        "World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}",
                  attr="Tiles &copy; Esri &mdash; Esri, DeLorme, NAVTEQ")
mapa.get_root().header.add_child(folium.Element(
    "<style>.leaflet-tile-pane{filter:grayscale(100%) brightness(107%) contrast(88%);}</style>"))
HeatMap(list(zip(celdas["lat"], celdas["lon"], celdas["bosque_base_ha"] / celdas["bosque_base_ha"].max())),
        name="Bosque base (contexto)", radius=9, blur=14, max_zoom=9, min_opacity=.15, show=False,
        gradient={0.2: "#2A3327", 0.6: "#4C5A4E", 1.0: "#1E6B45"}).add_to(mapa)
HeatMap(list(zip(con_def["lat"], con_def["lon"], con_def["area_def_total_ha"] / con_def["area_def_total_ha"].max())),
        name="Hectáreas con alerta (mapa de calor)", radius=11, blur=16, max_zoom=9, min_opacity=.25).add_to(mapa)
escala = cm.LinearColormap(["#FBE3CE", "#E2905A", "#B8562D", "#7A3319"],
                           vmin=float(np.log1p(con_def["area_def_total_ha"].min())),
                           vmax=float(np.log1p(con_def["area_def_total_ha"].max())))
escala.caption = "Hectáreas con alerta acumuladas por celda (escala logarítmica)"
mapa.add_child(escala)
racimo = MarkerCluster(name=f"Los {FOCOS_EN_EL_MAPA} focos más grandes")
for f in con_def.sort_values("area_def_total_ha", ascending=False).head(FOCOS_EN_EL_MAPA).itertuples():
    color = escala(np.log1p(f.area_def_total_ha))
    folium.CircleMarker(
        location=(f.lat, f.lon), radius=float(np.clip(3 + np.sqrt(f.area_def_total_ha) * 1.4, 3, 16)),
        color=color, fill=True, fill_color=color, fill_opacity=.85, weight=1,
        popup=folium.Popup(html=(f"<b>{f.cell_id}</b> · {f.departamento}<br>"
                                 f"Bosque base: {f.bosque_base_ha:,.0f} ha<br>"
                                 f"Con alerta: <b>{f.area_def_total_ha:,.1f} ha</b> "
                                 f"({100 * f.tasa_total:.2f} % del bosque)<br>"
                                 f"Meses con alerta: {int(f.meses_con_evento)}<br>"
                                 f"Último: {f.ultimo_evento:%Y-%m}"), max_width=260)).add_to(racimo)
racimo.add_to(mapa)
folium.LayerControl(collapsed=False).add_to(mapa)
Fullscreen(position="topleft").add_to(mapa)
mapa

---
## Anexo: sacar o renovar la API key de GFW

La API key de GFW es gratuita y **vence al año**. Si no tienes una, o si la celda 6 dice que GFW la rechazó,
sigue estos dos pasos. **Corre solo estas celdas**, una por una (no "Ejecutar todo"), porque el resto del
notebook necesita la llave para funcionar.

Estas celdas no hacen nada mientras tengan algún `<<LLENAR ...>>`: así no mandan nada a GFW si alguien
corre todo el notebook.

**Paso A, una sola vez por correo:** registra un correo en GFW. GFW manda a ese correo una contraseña
temporal.

In [ ]:
import requests

gfw_nombre = "<<LLENAR: tu nombre>>"
gfw_correo = "<<LLENAR: tu correo>>"

if "<<LLENAR" in gfw_nombre + gfw_correo:
    print("Paso A: llena gfw_nombre y gfw_correo en esta celda y vuelve a correrla.")
else:
    r = requests.post("https://data-api.globalforestwatch.org/auth/sign-up",
                      json={"name": gfw_nombre, "email": gfw_correo}, timeout=30)
    print(r.status_code, r.text[:300])
    print("Revisa el correo: GFW manda una contraseña temporal. Con ella, corre el paso B.")

**Paso B, cada vez que haga falta una llave:** con el correo y la contraseña, se pide un token y con el token
una API key nueva. Cópiala en `GFW_API_KEY`, en la celda 1. Si GFW cambió la forma de su respuesta, la celda
muestra la respuesta completa para buscar la llave a mano.

In [ ]:
gfw_correo = "<<LLENAR: tu correo>>"
gfw_contrasena = "<<LLENAR: la contraseña que llegó al correo>>"

if "<<LLENAR" in gfw_correo + gfw_contrasena:
    print("Paso B: llena gfw_correo y gfw_contrasena en esta celda y vuelve a correrla.")
else:
    r = requests.post("https://data-api.globalforestwatch.org/auth/token",
                      data={"username": gfw_correo, "password": gfw_contrasena},
                      headers={"Content-Type": "application/x-www-form-urlencoded"}, timeout=30)
    r.raise_for_status()
    respuesta = r.json()
    token = (respuesta.get("data") or {}).get("access_token") or respuesta.get("access_token")
    if not token:
        raise RuntimeError(f"No vino el token en la respuesta: {respuesta}")
    r = requests.post("https://data-api.globalforestwatch.org/auth/apikey",
                      json={"alias": f"observatorio-{pd.Timestamp.today():%Y%m%d}",
                            "organization": "Observatorio Financiero Rural",
                            "email": gfw_correo, "domains": []},
                      headers={"Authorization": f"Bearer {token}"}, timeout=30)
    r.raise_for_status()
    respuesta = r.json()
    llave = (respuesta.get("data") or {}).get("api_key") or respuesta.get("api_key")
    if llave:
        print("Tu API key nueva. Cópiala en GFW_API_KEY, en la celda 1:", llave)
    else:
        print("No vino la llave donde se esperaba. Respuesta completa:", respuesta)